# Small-scale adaptation: From Noise to Clarity

**Original paper:** Junbao Huang et al., *From Noise to Clarity: Filtering Real and LLM-Generated Samples for Enhanced Intent Detection*, Findings of EMNLP 2025, pp. 21736–21746.

This adaptation studies the paper's core idea—quality filtering before intent training—without claiming its topic models, LLM data, or published Banking-77 results. It creates deterministic character/word-corrupted training utterances, compares training on all noisy samples against a confidence-and-similarity filter, and evaluates on the untouched official Banking77 test split.

In [1]:
import csv, io, json, random, re, urllib.request
from pathlib import Path
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier

SEED = 42
random.seed(SEED); np.random.seed(SEED)
DATA_URL = "https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/master/banking_data/train.csv"
TEST_URL = "https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/master/banking_data/test.csv"
CLASSES = ["card_arrival", "card_linking", "exchange_rate", "card_payment_wrong_exchange_rate", "extra_charge_on_statement", "pending_cash_withdrawal", "fiat_currency_support", "card_delivery_estimate", "automatic_top_up", "card_not_working"]
# Keep a compact, public, reproducible subset. If a category is absent in a future mirror, fail clearly.
def load_rows(url):
    raw = urllib.request.urlopen(url, timeout=30).read().decode("utf-8")
    return list(csv.DictReader(io.StringIO(raw)))
train_rows = [r for r in load_rows(DATA_URL) if r["category"] in CLASSES]
test_rows = [r for r in load_rows(TEST_URL) if r["category"] in CLASSES]
if set(r["category"] for r in train_rows) != set(CLASSES):
    raise RuntimeError("The public Banking77 mirror does not contain the expected compact class subset")
X_all = np.array([r["text"] for r in train_rows], dtype=object)
y_all = np.array([r["category"] for r in train_rows], dtype=object)
X_test = np.array([r["text"] for r in test_rows], dtype=object)
y_test = np.array([r["category"] for r in test_rows], dtype=object)
# The official test file is held out. The train file is split again for a deterministic validation-style adaptation.
X_train, X_dev, y_train, y_dev = train_test_split(X_all, y_all, test_size=0.25, random_state=SEED, stratify=y_all)

def model():
    return Pipeline([("tfidf", TfidfVectorizer(lowercase=True, ngram_range=(1,2), min_df=1, sublinear_tf=True)), ("clf", LogisticRegression(max_iter=400, random_state=SEED))])
def score(name, clf, Xtr, ytr, Xev=X_test, yev=y_test):
    clf.fit(Xtr, ytr); pred=clf.predict(Xev)
    return {"condition": name, "accuracy": round(float(accuracy_score(yev,pred)), 6), "macro_f1": round(float(f1_score(yev,pred,average="macro")), 6), "n_train": int(len(Xtr)), "n_test": int(len(yev))}

In [2]:
def corrupt(text):
    words=text.split()
    if len(words)>4:
        words=words[:1]+words[2:]  # deterministic word deletion
    return re.sub(r"[aeiou]", "", " ".join(words), count=1)

X_noisy=np.array([corrupt(x) for x in X_train],dtype=object)
# A frozen TF-IDF similarity to the clean class centroid is a small, transparent quality signal.
vec=TfidfVectorizer(lowercase=True, ngram_range=(1,2), min_df=1)
Z=vec.fit_transform(np.concatenate([X_train,X_noisy]))
Zclean=Z[:len(X_train)]; Znoise=Z[len(X_train):]
centroids={c:Zclean[np.where(y_train==c)[0]].mean(axis=0) for c in CLASSES}
quality=[]
for i,c in enumerate(y_train):
    quality.append(float(Znoise[i].dot(centroids[c].T).__array__()[0,0]))
quality=np.array(quality)
threshold=float(np.quantile(quality,0.35))
keep=quality>=threshold
X_filtered=np.concatenate([X_train, X_noisy[keep]])
y_filtered=np.concatenate([y_train, y_train[keep]])
results=[]
results.append(score("clean baseline", model(), X_train, y_train))
results.append(score("all deterministic noisy additions", model(), np.concatenate([X_train,X_noisy]), np.concatenate([y_train,y_train])))
results.append(score("similarity-filtered noisy additions", model(), X_filtered, y_filtered))
out={"dataset":"Banking77 compact 10-class subset","noise":"one deterministic deletion plus vowel removal","threshold":threshold,"kept_noisy":int(keep.sum()),"results":results}
Path("results.json").write_text(json.dumps(out,indent=2))
print(json.dumps(out,indent=2))

{
  "dataset": "Banking77 compact 10-class subset",
  "noise": "one deterministic deletion plus vowel removal",
  "threshold": 0.06033578298077251,
  "kept_noisy": 661,
  "results": [
    {
      "condition": "clean baseline",
      "accuracy": 0.95,
      "macro_f1": 0.950371,
      "n_train": 1017,
      "n_test": 400
    },
    {
      "condition": "all deterministic noisy additions",
      "accuracy": 0.9575,
      "macro_f1": 0.95797,
      "n_train": 2034,
      "n_test": 400
    },
    {
      "condition": "similarity-filtered noisy additions",
      "accuracy": 0.9575,
      "macro_f1": 0.957773,
      "n_train": 1678,
      "n_test": 400
    }
  ]
}
